# Multi-agent internals

Small checks between steps: is the agent looping, is the handoff complete, did the step match the plan, which answer is supported.


In [ ]:
import sys
from datetime import date
from pathlib import Path
import json
import re
import statistics

ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from langchain_typesafe import Choice, Noul, NoulCriteria, Score
from jev_examples.settings import ask, ask_many, draft, jev_model, openai_ready, show, typesafe_ready
from jev_examples.sample_data import (
    corpus_docs,
    customers,
    emails,
    load_json,
    lookup_order,
    open_incidents,
    order,
    products,
    read_text,
    ticket,
    tickets,
)

print("Jev model:", jev_model())
print("Jev key set:", typesafe_ready())
print("OpenAI key set:", openai_ready())


## 37. Stop a loop

Three identical lookups in a row, and no new result, is a stall. The goal is still the A-104 refund.


In [ ]:
if not typesafe_ready():
    print("skipped: set TYPESAFE_API_KEY in .env")
else:
    agent = load_json("agent.json")
    response = ask(
        {
            "goal": agent["current_goal"],
            "previous_steps": agent["loop_steps"][:-1],
            "latest_step": agent["loop_steps"][-1],
            "latest_result": "",
        },
        {
            "is_repeating": Noul(instructions="Is `latest_step` the same action as one of `previous_steps`?"),
            "made_progress": Noul(instructions="Does `latest_result` move closer to `goal` than the earlier steps did?"),
        },
    )
    show(response)
    if response.nouls["is_repeating"].noul > 0.6 and response.nouls["made_progress"].noul < 0.4:
        route = "stop"
    else:
        route = "continue"
    print("route:", route)


**What you should see.** The repeated `lookup_order A-104` with an empty result should stop.


## 38. Is the handoff complete?

The receiving agent needs a goal, the facts, and a way to know it is finished.


In [ ]:
if not typesafe_ready():
    print("skipped: set TYPESAFE_API_KEY in .env")
else:
    questions = {
        "has_goal": Noul(instructions="Does `message` state what the receiving agent must accomplish?"),
        "has_context": Noul(instructions="Does `message` include the ids or amounts the receiving agent needs?"),
        "has_done_criteria": Noul(instructions="Does `message` say how to know the work is finished?"),
    }
    for item in load_json("agent.json")["handoffs"]:
        response = ask(item, questions)
        show(response)
        missing = [name for name, answer in response.nouls.items() if answer.noul < 0.55]
        print(item["to"], "->", "send" if not missing else missing)


**What you should see.** The refund handoff that names $49 and A-104 should send. 'Handle the tent thing' should come back with missing pieces.


## 39. Did the step do what the plan said?

Compare the plan line with what actually happened before taking the next step.


In [ ]:
if not typesafe_ready():
    print("skipped: set TYPESAFE_API_KEY in .env")
else:
    questions = {
        "status": Choice(
            instructions="Comparing `plan_step` with `what_happened`, what is the status?",
            criteria={
                "done": "The step was completed as written",
                "partial": "Some of the step was done",
                "diverged": "Something else was done",
            },
        ),
        "safe_to_continue": Noul(instructions="Is it safe to proceed to `next_plan_step` without a person?"),
    }
    for item in load_json("agent.json")["plan_steps"]:
        response = ask(item, questions)
        show(response)
        status = response.choices["status"]
        if status.choice == "diverged" or response.nouls["safe_to_continue"].noul < 0.6:
            route = "human_gate"
        else:
            route = "next_step"
        print("route:", route)


**What you should see.** The lookup that returned two $49 lines should continue. The marketing email should go to a person.


## 40. Two specialists disagree

Pick the answer the evidence supports. If neither is supported, escalate.


In [ ]:
if not typesafe_ready():
    print("skipped: set TYPESAFE_API_KEY in .env")
else:
    item = load_json("agent.json")["subagent_answers"]
    response = ask(
        item,
        {
            "better_supported": Choice(
                instructions="Which answer is better supported by `evidence`?",
                criteria={"a": "Answer A", "b": "Answer B", "neither": "Neither is supported"},
            ),
            "contradict": Noul(instructions="Do `a` and `b` make incompatible claims?"),
        },
    )
    show(response)
    pick = response.choices["better_supported"]
    if pick.choice == "neither" or pick.confidence < 0.55:
        route = "escalate"
    else:
        route = pick.choice
    print("route:", route)


**What you should see.** Answer A (the $49 bottle) matches the evidence. Answer B (a $189 tent) does not. They contradict each other.


## 41. Is the tool result usable?

An error page or a login wall should not be treated as the order.


In [ ]:
if not typesafe_ready():
    print("skipped: set TYPESAFE_API_KEY in .env")
else:
    questions = {
        "is_error_page": Noul(instructions="Is `output` an error, a login wall, or an empty page rather than the data?"),
        "matches_call": Noul(instructions="Is `output` plausibly the result of `call`?"),
    }
    for item in load_json("agent.json")["tool_results"]:
        response = ask(item, questions)
        show(response)
        if response.nouls["is_error_page"].noul > 0.6 or response.nouls["matches_call"].noul < 0.4:
            route = "retry"
        else:
            route = "use"
        print("route:", route)


**What you should see.** The shipped-tent sentence should be used. The session-expired page should retry.


## 42. The request could mean two things

If more than one reading is plausible, say so instead of guessing. 'Can you check the pack?' is the sample.


In [ ]:
if not typesafe_ready():
    print("skipped: set TYPESAFE_API_KEY in .env")
else:
    agent = load_json("agent.json")
    questions = {
            "meant_%s" % key: Noul(instructions="Could `request` mean: %s?" % text)
            for key, text in agent["interpretations"].items()
        }
    response = ask({"request": agent["ambiguous"]}, questions)
    show(response)
    likely = [key for key in agent["interpretations"] if response.nouls["meant_%s" % key].noul > 0.45]
    print("route:", "clarify" if len(likely) != 1 else likely[0], likely)


**What you should see.** Both readings of 'the pack' can be plausible, so the route should clarify. If one is clearly higher, that single reading is fine too.
